# Tratamento dos Dados — Casos por ano de diagóstico, faixa etária, escolaridade e raça cor


---
## 1. Imports

In [1]:
from __future__ import annotations

import re
import unicodedata
from io import StringIO
from pathlib import Path

import pandas as pd

---
## 2. Configurações Gerais


In [2]:
# ── Caminhos ─────────────────────────────────────────────────────────────────
BASE_VARIAVEIS = Path("/home/anna/tcc/Limpeza e Integração/Casos")
OUT_DIR        = Path("/home/anna/tcc/Seleção e Transformação/Casos")

# ── Subpastas de cada dimensão ────────────────────────────────────────────────
DIR_ANO_DIAG    = BASE_VARIAVEIS / "Ano Diagnóstico" / "Dados"
DIR_ESCOLARIDADE = BASE_VARIAVEIS / "Escolaridade"   / "Dados"
DIR_FAIXA_ETARIA = BASE_VARIAVEIS / "Faixa Etária"   / "Dados"
DIR_RACA_COR     = BASE_VARIAVEIS / "Raça-Cor"        / "Dados"

# ── Nomes dos arquivos de saída ───────────────────────────────────────────────
SAIDA_ANO_DIAG    = OUT_DIR / "tabela_casos_por_ano_de_diagnóstico.csv"
SAIDA_ESCOLARIDADE = OUT_DIR / "tabela_casos_por_escolaridade.csv"
SAIDA_FAIXA_ETARIA = OUT_DIR / "tabela_casos_por_faixa_etária.csv"
SAIDA_RACA_COR     = OUT_DIR / "tabela_casos_por_raça_e_cor.csv"

# ── Faixas etárias padronizadas (destino após agrupamento) ────────────────────
FAIXA_TARGET = [
    "0-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70 OU MAIS",   # unifica 70-79 e 80 ou mais
]

# ── Padronização dos nomes de categorias (CAIXA ALTA + acentuação correta) ────
CATEGORIA_PADRAO: dict[str, str] = {
    "fundamental completo":     "FUNDAMENTAL COMPLETO",
    "fundamental incompleto":   "FUNDAMENTAL INCOMPLETO",
    "nenhuma":                  "NENHUMA",
    "nivel medio":              "NÍVEL MÉDIO",
    "nivel superior completo":  "NÍVEL SUPERIOR COMPLETO",
    "nivel superior incompleto": "NÍVEL SUPERIOR INCOMPLETO",
    "sem informacao":           "SEM INFORMAÇÃO",
    "amarela":                  "AMARELA",
    "branca":                   "BRANCA",
    "indigena":                 "INDÍGENA",
    "parda":                    "PARDA",
    "preta":                    "PRETA",
}

# ── Intervalo válido de anos de diagnóstico ───────────────────────────────────
ANO_MIN_VALIDO = 1985
ANO_MAX_VALIDO = 2023

OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 3. Funções de Normalização de Texto

Funções utilitárias para padronizar texto, nomes de colunas e extrair informações dos campos brutos.  
Todas as demais etapas dependem dessas funções — por isso ficam definidas primeiro.

In [3]:
def normalize_text(value: str) -> str:
    """Remove acentos, converte para minúsculas e colapsa espaços múltiplos."""
    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r"\s+", " ", text)
    return text


def clean_col(col: str) -> str:
    """Remove BOM, aspas e normaliza o nome de coluna sem alterar maiúsculas."""
    col = str(col).replace("\ufeff", "").replace('"', "").strip()
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    col = re.sub(r"\s+", " ", col).strip()
    return col


def parse_municipio_codigo(procedencia: str) -> tuple[str | None, int | None]:
    """
    Extrai município e código IBGE do campo de procedência.
    Formato esperado: 'MG - 3100000 - Nome do Município'
    Retorna (municipio_normalizado, codigo_ibge) ou (None, None) se não casar.
    """
    text  = str(procedencia).replace('"', "").strip()
    match = re.match(r"^MG\s*-\s*(\d{7})\s*-\s*(.+)$", text, flags=re.IGNORECASE)
    if not match:
        return None, None
    codigo    = int(match.group(1))
    municipio = normalize_text(match.group(2))
    return municipio, codigo


def extract_year(file_name: str) -> int:
    """Extrai o ano (19xx ou 20xx) do nome do arquivo. Lança ValueError se não encontrar."""
    match = re.search(r"(19\d{2}|20\d{2})", file_name)
    if not match:
        raise ValueError(f"Ano não encontrado no nome do arquivo: {file_name}")
    return int(match.group(1))


---
## 4. Funções de Leitura dos Arquivos Brutos

Os arquivos brutos do RHC/INCA chegam como CSVs com encoding `latin1`, separador `;` e
linhas de cabeçalho não padronizadas. A função `read_raw_table` localiza automaticamente
a linha de cabeçalho real (que contém a coluna de procedência) e descarta as linhas anteriores.

In [4]:
def read_raw_table(file_path: Path) -> pd.DataFrame:
    """
    Lê um arquivo bruto do RHC/INCA (latin1, sep=';') localizando o cabeçalho real.
    Adiciona as colunas MUNICIPIOS e CODIGO a partir do campo de procedência.
    Descarta linhas sem código IBGE válido.
    """
    content    = file_path.read_text(encoding="latin1", errors="ignore").splitlines()
    header_idx = None
    for idx, line in enumerate(content):
        if ";" in line and "proced" in normalize_text(line):
            header_idx = idx
            break
    if header_idx is None:
        raise ValueError(f"Cabeçalho não encontrado em: {file_path}")

    table_text = "\n".join(content[header_idx:])
    df = pd.read_csv(StringIO(table_text), sep=";", engine="python", encoding="latin1")
    df.columns = [clean_col(c) for c in df.columns]

    # Remove colunas sem nome (campos em branco no cabeçalho viram "Unnamed: N" pelo pandas)
    unnamed = [c for c in df.columns if clean_col(c) == "" or re.match(r"(?i)^unnamed", c)]
    if unnamed:
        df = df.drop(columns=unnamed)

    # Extrai município e código da primeira coluna (procedência)
    col_proc = df.columns[0]
    parsed   = df[col_proc].apply(parse_municipio_codigo)
    df["MUNICIPIOS"] = parsed.map(lambda x: x[0])
    df["CODIGO"]     = parsed.map(lambda x: x[1])

    # Mantém apenas linhas com código IBGE válido
    df = df[df["CODIGO"].notna()].copy()
    df["CODIGO"]     = df["CODIGO"].astype("Int64")
    df["MUNICIPIOS"] = df["MUNICIPIOS"].astype(str)

    return df


def numeric_columns(df: pd.DataFrame, skip: set[str]) -> list[str]:
    """Retorna colunas numéricas do df, ignorando as listadas em skip e a coluna 'total'."""
    cols = []
    for col in df.columns:
        if col in skip:
            continue
        if normalize_text(col) in {"total", ""}:
            continue
        cols.append(col)
    return cols


def sorted_files(directory: Path, pattern: str) -> list[Path]:
    """Retorna arquivos do diretório que casam com pattern, ordenados por ano no nome."""
    files = list(directory.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em: {directory}/{pattern}")
    return sorted(files, key=lambda p: extract_year(p.name))


---
## 5. Mapeamento de Faixas Etárias

Os arquivos brutos usam faixas de 5 em 5 anos (0-4, 5-9, 10-14 …). Esta função
agrupa essas faixas nas 8 categorias padronizadas definidas em `FAIXA_TARGET`,
unificando 70-79 e 80+ em **"70 ou mais"**.

In [5]:
def map_faixa(col: str) -> str | None:
    c = normalize_text(col).replace(" ", "")
    mapa = {
        "00-04": "0-9",  "0-4":  "0-9",  "05-09": "0-9",  "5-9":  "0-9",
        "10-14": "10-19", "15-19": "10-19",
        "20-24": "20-29", "25-29": "20-29",
        "30-34": "30-39", "35-39": "30-39",
        "40-44": "40-49", "45-49": "40-49",
        "50-54": "50-59", "55-59": "50-59",
        "60-64": "60-69", "65-69": "60-69",
        "70-74": "70 OU MAIS", "75-79": "70 OU MAIS", "70-79": "70 OU MAIS",
        "80-84": "70 OU MAIS", "85+":   "70 OU MAIS", "85-89": "70 OU MAIS",
        "90+":   "70 OU MAIS", "80oumais": "70 OU MAIS", "80+": "70 OU MAIS",
    }
    return mapa.get(c)

---
## 6. Funções de Agregação

### 6.1 Ano de Diagnóstico
Agrega todos os arquivos anuais em uma única tabela com uma coluna por ano (1985–2023),
somando os casos por município.

In [6]:
def renomear_coluna_ano_diagnostico(col: str) -> str:
    """
    Renomeia uma coluna bruta de Ano de Diagnóstico para o padrão <ano>_<categoria>:
    coluna de ano vira "<ano>_DIAGNOSTICO" (categoria única desta variável, já que
    não há sub-divisão por categoria); a coluna de casos sem ano de diagnóstico
    conhecido vira "ANO_SEM-INFORMAÇÃO" (não tem ano real para prefixar, por isso
    usa a palavra "ANO" no lugar do número, mantendo o mesmo formato <algo>_<categoria>).
    """
    limpo = clean_col(col)
    if re.fullmatch(r"\d{4}", limpo):
        return f"{limpo}_DIAGNOSTICO"
    categoria = normalize_categoria(col)
    if categoria == "SEM INFORMAÇÃO":
        return "ANO_SEM-INFORMAÇÃO"
    return categoria


def aggregate_ano_diagnostico(raw_files: list[Path]) -> pd.DataFrame:
    """
    Lê todos os arquivos de Ano de Diagnóstico e os concatena em uma única tabela.
    Cada coluna de ano vira "<ano>_DIAGNOSTICO" (padrão <ano>_<categoria>, igual ao
    das demais dimensões); linhas = municípios. Colunas de ano fora de
    [ANO_MIN_VALIDO, ANO_MAX_VALIDO] são descartadas (mesmo comportamento de sempre
    -- os arquivos brutos às vezes trazem colunas de anos fora do período válido).
    """
    frames: list[pd.DataFrame] = []
    for file_path in raw_files:
        df   = read_raw_table(file_path)
        cols = numeric_columns(df, {df.columns[0], "MUNICIPIOS", "CODIGO"})
        rename_map = {col: renomear_coluna_ano_diagnostico(col) for col in cols}
        current = df[["MUNICIPIOS", "CODIGO", *cols]].rename(columns=rename_map)
        frames.append(current)

    long_df    = pd.concat(frames, ignore_index=True)
    value_cols = [c for c in long_df.columns if c not in {"MUNICIPIOS", "CODIGO"}]
    for col in value_cols:
        long_df[col] = pd.to_numeric(long_df[col], errors="coerce")

    agg = long_df.groupby(["MUNICIPIOS", "CODIGO"], as_index=False)[value_cols].sum(min_count=1)

    # Ordena colunas: anos primeiro (por ano, dentro do período válido), depois
    # demais (ex.: ANO_SEM-INFORMAÇÃO). Colunas "<ano>_DIAGNOSTICO" fora do
    # período válido são descartadas por completo, não só reordenadas.
    ano_cols = sorted(
        [
            c for c in value_cols
            if re.fullmatch(r"\d{4}_DIAGNOSTICO", str(c))
            and ANO_MIN_VALIDO <= int(str(c)[:4]) <= ANO_MAX_VALIDO
        ],
        key=lambda c: int(str(c)[:4]),
    )
    tail_cols = sorted([
        c for c in value_cols
        if c not in ano_cols and not re.fullmatch(r"\d{4}_DIAGNOSTICO", str(c))
    ])
    ordered   = ["MUNICIPIOS", "CODIGO", *ano_cols, *tail_cols]

    return agg[ordered].rename(columns={"MUNICIPIOS": "MUNICÍPIO", "CODIGO": "CÓDIGO IBGE"})


### 6.2 Dimensões categóricas (Escolaridade, Raça/Cor e Faixa Etária)

Agrega os arquivos anuais em formato **wide**: cada coluna recebe o nome `YYYY_categoria`.
Para Faixa Etária, as colunas de 5 em 5 anos são agrupadas nas 8 faixas padronizadas.

In [7]:
def aggregate_wide_by_year(
    raw_files: list[Path],
    category_mapper=None,
    *,
    faixa_mode: bool = False,
) -> pd.DataFrame:
    """
    Itera sobre os arquivos brutos (um por ano) e monta uma tabela wide com colunas
    no formato YYYY_categoria.

    Parâmetros
    ----------
    raw_files       : lista de arquivos brutos ordenados por ano
    category_mapper : função que recebe o nome bruto da coluna e retorna o nome padronizado
                      (ignorado quando faixa_mode=True)
    faixa_mode      : se True, usa map_faixa() para agrupar faixas de 5 em 5 anos nas 8 target
    """
    result: pd.DataFrame | None = None

    for file_path in sorted(raw_files, key=lambda p: extract_year(p.name)):
        year = extract_year(file_path.name)
        df   = read_raw_table(file_path)
        cols = numeric_columns(df, {df.columns[0], "MUNICIPIOS", "CODIGO"})

        work = df[["MUNICIPIOS", "CODIGO", *cols]].copy()
        for col in cols:
            work[col] = pd.to_numeric(work[col], errors="coerce")

        if faixa_mode:
            # Agrupa colunas brutas nas 8 faixas padronizadas
            grouped_cols: dict[str, list[str]] = {k: [] for k in FAIXA_TARGET}
            for col in cols:
                mapped = map_faixa(col)
                if mapped is not None:
                    grouped_cols[mapped].append(col)

            current = work[["MUNICIPIOS", "CODIGO"]].copy()
            for target in FAIXA_TARGET:
                src = grouped_cols[target]
                current[f"{year}_{target}"] = work[src].sum(axis=1, min_count=1) if src else pd.NA
        else:
            renamed = {col: f"{year}_{category_mapper(col)}" for col in cols}
            current = work[["MUNICIPIOS", "CODIGO", *cols]].rename(columns=renamed)

        result = current if result is None else result.merge(current, on=["MUNICIPIOS", "CODIGO"], how="outer")

    if result is None:
        raise ValueError("Nenhum arquivo processado.")

    return result.rename(columns={"MUNICIPIOS": "MUNICÍPIO", "CODIGO": "CÓDIGO IBGE"})


def normalize_categoria(col: str) -> str:
    """Padroniza nome de categoria em CAIXA ALTA com acentuação correta (ver CATEGORIA_PADRAO)."""
    chave = normalize_text(clean_col(col))
    return CATEGORIA_PADRAO.get(chave, clean_col(col).upper())


---
## 6.3 Total por ano

Adiciona, ao final de cada tabela wide, uma coluna `TOTAL_YYYY` para cada ano presente
(seja o próprio ano, como em Ano de Diagnóstico, seja o conjunto `YYYY_categoria`,
como em Escolaridade/Faixa Etária/Raça-Cor). Cada `TOTAL_YYYY` é a soma das colunas
daquele ano **na própria linha** (ou seja, o total do município naquele ano, somando
as categorias) — não o total do estado inteiro.

In [8]:
def add_total_ano_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada ano presente nas colunas (no formato 'YYYY_categoria'),
    adiciona ao final da tabela uma coluna 'YYYY_TOTAL' com a soma das
    colunas daquele ano NA PRÓPRIA LINHA (total do município naquele ano,
    somando as categorias) — não o total do estado inteiro.
    """
    df = df.copy()
    anos = sorted({
        match.group(1)
        for col in df.columns
        if (match := re.match(r"^(\d{4})(?:_|$)", str(col)))
    })
    for ano in anos:
        cols_ano = [c for c in df.columns if re.match(rf"^{ano}(_|$)", str(c))]
        df[f"{ano}_TOTAL"] = df[cols_ano].sum(axis=1, min_count=1)
    return df

---
## 6.4 Regional de Saúde

Adiciona a coluna `REGIONAL DE SAÚDE` a cada tabela, via join pelo Código IBGE com
`tabela_municipios_regionais.csv` — mesma base de referência usada pelas outras
etapas do projeto para mapear município → Regional.

In [9]:
CAMINHO_MUNICIPIOS_REGIONAIS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")


def carregar_mapa_regional() -> pd.Series:
    """Carrega o mapa CÓDIGO IBGE → REGIONAL DE SAÚDE a partir da tabela de municípios."""
    mapa = pd.read_csv(CAMINHO_MUNICIPIOS_REGIONAIS, encoding="utf-8-sig")
    mapa["CÓDIGO IBGE"] = pd.to_numeric(mapa["CÓDIGO IBGE"], errors="coerce").astype("Int64")
    return mapa.set_index("CÓDIGO IBGE")["REGIONAL NORMALIZADA"].rename("REGIONAL DE SAÚDE")


def add_regional_column(df: pd.DataFrame) -> pd.DataFrame:
    """Adiciona REGIONAL DE SAÚDE logo após CÓDIGO IBGE, via join pelo código IBGE."""
    mapa_regional = carregar_mapa_regional()
    df = df.copy()
    df["CÓDIGO IBGE"] = pd.to_numeric(df["CÓDIGO IBGE"], errors="coerce").astype("Int64")
    df["REGIONAL DE SAÚDE"] = df["CÓDIGO IBGE"].map(mapa_regional)

    sem_regional = int(df["REGIONAL DE SAÚDE"].isna().sum())
    if sem_regional > 0:
        print(f"   ⚠️ {sem_regional} município(s) sem Regional de Saúde correspondente.")

    colunas = df.columns.tolist()
    colunas.remove("REGIONAL DE SAÚDE")
    posicao = colunas.index("CÓDIGO IBGE") + 1
    colunas.insert(posicao, "REGIONAL DE SAÚDE")
    return df[colunas]

---
## 7. Pipeline de Tratamento

Executa o pipeline completo em 5 etapas:
1. Localiza os arquivos brutos de cada dimensão
2. Agrega Ano de Diagnóstico
3. Agrega Escolaridade, Faixa Etária e Raça/Cor (formato wide)
4. Adiciona `YYYY_TOTAL` ao final de cada tabela
5. Adiciona `REGIONAL DE SAÚDE` e salva os CSVs de saída (um arquivo por dimensão)

In [10]:
# ── Etapa 1: Localizar arquivos ───────────────────────────────────────────────
print("Arquivos brutos:")
files_ano    = sorted_files(DIR_ANO_DIAG,     "AnoDiagnostico-*.csv")
files_esc    = sorted_files(DIR_ESCOLARIDADE,  "Escolaridade-*.csv")
files_faixa  = sorted_files(DIR_FAIXA_ETARIA,  "Faixa-Etaria-*.csv")
files_raca   = sorted_files(DIR_RACA_COR,      "RacaCor-*.csv")

print(f"   Ano Diagnóstico : {len(files_ano)} arquivo(s)")
print(f"   Escolaridade    : {len(files_esc)} arquivo(s)")
print(f"   Faixa Etária    : {len(files_faixa)} arquivo(s)")
print(f"   Raça/Cor        : {len(files_raca)} arquivo(s)")

Arquivos brutos:
   Ano Diagnóstico : 38 arquivo(s)
   Escolaridade    : 38 arquivo(s)
   Faixa Etária    : 38 arquivo(s)
   Raça/Cor        : 38 arquivo(s)


In [11]:
# ── Etapa 2: Ano de Diagnóstico ──────────────────────────────────────────────
print("\nProcessando Ano de Diagnóstico...")
df_ano_diag = aggregate_ano_diagnostico(files_ano)
print(f"   Shape: {df_ano_diag.shape} | Municípios: {df_ano_diag['MUNICÍPIO'].nunique()}")
df_ano_diag.head(3)


Processando Ano de Diagnóstico...


   Shape: (853, 42) | Municípios: 853


,MUNICÍPIO,CÓDIGO IBGE,1985_DIAGNOSTICO,1986_DIAGNOSTICO,1987_DIAGNOSTICO,1988_DIAGNOSTICO,1989_DIAGNOSTICO,1990_DIAGNOSTICO,1991_DIAGNOSTICO,1992_DIAGNOSTICO,...,2015_DIAGNOSTICO,2016_DIAGNOSTICO,2017_DIAGNOSTICO,2018_DIAGNOSTICO,2019_DIAGNOSTICO,2020_DIAGNOSTICO,2021_DIAGNOSTICO,2022_DIAGNOSTICO,2023_DIAGNOSTICO,ANO_SEM-INFORMAÇÃO
0,abadia dos dourados,3100104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2.0,2.0,1.0,0.0,1.0,0.0,2.0,0.0
1,abaete,3100203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.0,7.0,3.0,2.0,3.0,5.0,6.0,14.0,9.0,4.0
2,abre campo,3100302,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,6.0,4.0,4.0,3.0,5.0,4.0,3.0,3.0,1.0


In [12]:
# ── Etapa 3a: Escolaridade ───────────────────────────────────────────────────
print("Processando Escolaridade...")
df_escolaridade = aggregate_wide_by_year(files_esc, normalize_categoria)
print(f"   Shape: {df_escolaridade.shape} | Municípios: {df_escolaridade['MUNICÍPIO'].nunique()}")

Processando Escolaridade...


   Shape: (853, 163) | Municípios: 853


In [13]:
# ── Etapa 3b: Faixa Etária ───────────────────────────────────────────────────
print("Processando Faixa Etária...")
df_faixa_etaria = aggregate_wide_by_year(files_faixa, faixa_mode=True)
print(f"   Shape: {df_faixa_etaria.shape} | Municípios: {df_faixa_etaria['MUNICÍPIO'].nunique()}")

Processando Faixa Etária...


   Shape: (853, 306) | Municípios: 853


In [14]:
# ── Etapa 3c: Raça/Cor ───────────────────────────────────────────────────────
print("Processando Raça/Cor...")
df_raca_cor = aggregate_wide_by_year(files_raca, normalize_categoria)
print(f"   Shape: {df_raca_cor.shape} | Municípios: {df_raca_cor['MUNICÍPIO'].nunique()}")

Processando Raça/Cor...


   Shape: (853, 150) | Municípios: 853


In [15]:
# ── Etapa 4: Total por ano ────────────────────────────────────────────────────
print("\nAdicionando YYYY_TOTAL por dimensão...")
df_ano_diag     = add_total_ano_columns(df_ano_diag)
df_escolaridade = add_total_ano_columns(df_escolaridade)
df_faixa_etaria = add_total_ano_columns(df_faixa_etaria)
df_raca_cor     = add_total_ano_columns(df_raca_cor)


Adicionando YYYY_TOTAL por dimensão...


In [16]:
# ── Etapa 4b: Regional de Saúde ────────────────────────────────────────────────
print("\nAdicionando REGIONAL DE SAÚDE por dimensão...")
df_ano_diag     = add_regional_column(df_ano_diag)
df_escolaridade = add_regional_column(df_escolaridade)
df_faixa_etaria = add_regional_column(df_faixa_etaria)
df_raca_cor     = add_regional_column(df_raca_cor)


Adicionando REGIONAL DE SAÚDE por dimensão...


In [17]:
# ── Etapa 5: Salvar CSVs ─────────────────────────────────────────────────────
print("\n💾 Salvando arquivos...")

df_ano_diag.to_csv(SAIDA_ANO_DIAG,     index=False)
df_escolaridade.to_csv(SAIDA_ESCOLARIDADE, index=False)
df_faixa_etaria.to_csv(SAIDA_FAIXA_ETARIA, index=False)
df_raca_cor.to_csv(SAIDA_RACA_COR,     index=False)


💾 Salvando arquivos...


---
## 8. Inspeção Pós-Processamento

Verificação rápida dos arquivos gerados: shape, primeiras linhas e contagem de valores ausentes.

In [18]:
from IPython.display import display, Markdown

tabelas = {
    "Ano de Diagnóstico": df_ano_diag,
    "Escolaridade"      : df_escolaridade,
    "Faixa Etária"      : df_faixa_etaria,
    "Raça/Cor"          : df_raca_cor,
}

for nome, df in tabelas.items():
    nulos = int(df.isnull().sum().sum())
    display(Markdown(f"**{nome}** — {df.shape[0]} linhas × {df.shape[1]} colunas | nulos totais: {nulos}"))
    display(df.head(2))

**Ano de Diagnóstico** — 853 linhas × 82 colunas | nulos totais: 598

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1985_DIAGNOSTICO,1986_DIAGNOSTICO,1987_DIAGNOSTICO,1988_DIAGNOSTICO,1989_DIAGNOSTICO,1990_DIAGNOSTICO,1991_DIAGNOSTICO,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,2.0,2.0,1.0,0.0,1.0,0.0,2.0
1,abaete,3100203,sete lagoas,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.0,6.0,7.0,3.0,2.0,3.0,5.0,6.0,14.0,9.0


**Escolaridade** — 853 linhas × 190 colunas | nulos totais: 54922

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1998_SEM INFORMAÇÃO,1999_NÍVEL MÉDIO,1999_SEM INFORMAÇÃO,2000_FUNDAMENTAL COMPLETO,2000_FUNDAMENTAL INCOMPLETO,2000_NENHUMA,2000_NÍVEL MÉDIO,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.0,1.0,1.0,2.0,1.0,NaN,NaN,2.0,2.0
1,abaete,3100203,sete lagoas,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,7.0,4.0,7.0,2.0,2.0,2.0,7.0,8.0,14.0,14.0


**Faixa Etária** — 853 linhas × 345 colunas | nulos totais: 175863

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1985_0-9,1985_10-19,1985_20-29,1985_30-39,1985_40-49,1985_50-59,1985_60-69,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,2.0,1.0,1.0,2.0,1.0,NaN,NaN,2.0,2.0
1,abaete,3100203,sete lagoas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,4.0,7.0,2.0,2.0,2.0,7.0,8.0,14.0,14.0


**Raça/Cor** — 853 linhas × 177 colunas | nulos totais: 53949

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1998_BRANCA,1998_PARDA,1998_PRETA,1998_SEM INFORMAÇÃO,1999_BRANCA,1999_PRETA,1999_SEM INFORMAÇÃO,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.0,1.0,1.0,2.0,1.0,NaN,NaN,2.0,2.0
1,abaete,3100203,sete lagoas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,4.0,7.0,2.0,2.0,2.0,7.0,8.0,14.0,14.0
